### Motivation behind multi attention heads 

A single self-attention computes one attention head which captures one kind of relationship. However, with multi attention heads, with each having its own Q, K, V projections and concatenated ouputs and each head operating in a smaller subsapce of dimension d_model / n_heads, which improves the expressive power.

Multi-head attention is the default for every transformer, but difference lies with how many heads and whether keys and values share projections (Grouped-Query Attention, Multi-Query Attention, Multi-head Latent Attention)

##### Variations in Attention Types 

* Variant 1: Multi-head Attention (MHA): Requires N K/V Heads
-- Used in GPT2/ BERT/ T5
* Variant 2: Multi-Query Attention (MQA): Requires 1 K/V Head (Re-use the same K/V across all heads)
-- Used in PaLM/ Falcon
* Variant 3: Grouped-Query Attention (GQA): Requires G K/V Heads (Reuses the K/V G times)
-- Used in LLama/ Qwen/ Mistral

* Variant 4: Multi-head latent (MLA) ~ Distinct concept from the other 3 Attention styles 
-- Used in Deepseek

In [1]:
import math
import random 
from typing import List 

In [7]:
class Matrix:
    __slots__ = ('rows', 'cols', 'data')

    def __init__(self, rows, cols, fill = 0.8, data = None):
        self.rows = rows
        self.cols = cols
        if data is not None:
            self.data = data 
        else:
            self.data = [fill] * (self.rows * self.cols)
    
    def get(self, i, j):
        return self.data[i * self.cols + j]
    
    def set(self, i, j, v):
        self.data[i * self.columns + j] = v
    
    def row(self, i):
        return self.data[i * self.cols : (i + 1) * self.cols]


In [19]:
def randn_matrix(rows, cols, rng, scale=None):
    if scale is None: 
        scale = math.sqrt(2.0 / (rows + cols))
    m = Matrix(rows, cols)
    for i in range(rows * cols):
        m.data[i] = rng.gauss(0.0, scale)
    return m

In [23]:
### Optimised version for multiplication

def matmul(A, B):
    assert A.cols == B.rows, f"{A.cols} != {B.rows}"
    out = Matrix(A.rows, B.cols)
    for i in range(A.rows):
        for k in range(A.cols):
            aik = A.get(i, k)
            if aik == 0.0:
                continue
            base_i = i * B.cols
            base_k = k * B.cols
            for j in range(B.cols):
                out.data[base_i + j] += aik * B.data[base_k + j]

    return out

## Textbook version for better understanding 
def matmul(A, B):
    assert A.cols == B.rows
    out = Matrix(A.rows, B.cols)
    for i in range(A.rows):          # row of A
        for j in range(B.cols):      # column of B
            s = 0.0
            for k in range(A.cols):  # summation dimension
                s += A.get(i, k) * B.get(k, j)
            out.set(i, j, s)

    return out

In [22]:
def transpose(A):
    out = Matrix(A.cols, A.rows)
    for i in range(A.rows):
        for j in range(A.cols):
            out.set(j, i, A.get(i,j))

    return out

In [24]:
def softmax_rows(A):
    out = Matrix(A.rows, A.cols)
    for i in range(A.rows):
        row = A.row(i)
        m = max(row)
        exps = [math.exp(x - m) for x in row]
        s = sum(exps)
        for j, e in enumerate(exps):
            out.set(i, j, e/s)
    
    return out

In [25]:
def scaled_dot_product_attention(Q, K, V):
    dk = Q.cols
    scale = 1.0 / sqrt(dk)
    scores = matmul(Q, transpose(K))
    for i in range(scores.rows * scores.cols):
        scores.data[i] *= scale
    weights = softmax_rows(scores)
    out = matmul(weights, V)
    return out, weights 

In [26]:
### Optimised version of multi-head attention - single matrix multiplication but subsequently broken down
###  into individuals heads

def split_heads(X, n_heads):
    assert X.cols % n_heads == 0, "d_model not divisible by n_heads"
    d_head = X.cols // n_heads
    heads = []
    for h in range(n_heads):
        H = Matrix(X.rows, d_head)
        for i in range(X.rows):
            for j in range(d_head):
                H.set(i, j, X.get(i, h * d_head + j))
        heads.append(H)
    
    return heads

def combine_heads(heads):
    n = heads[0].rows
    d_head = heads[0].cols
    d_model = d_head * len(heads)
    out = Matrix(n, d_model)
    for h, H in enumerate(heads):
        for i in range(n):
            for j in range(d_head):
                out.set(i, h * d_head + j, H.get(i,j))
    
    return out

In [ ]:
def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    Q = matmul(X, Wq)
    K = matmul(X, Wk)
    V = matmul(X, Wv)
    Qh = split_heads(Q, n_heads)
    Kh = split_heads(K, n_heads)
    Vh = split_heads(V, n_heads)
    
    head_outs = []
    per_head_weights = []
    for q, k, v in zip(Qh, Kh, Vh):
        o, w = scaled_dot_product_attention(q, k, v)
        head_outs.append(o)
        per_head_weights.append(w)
    concat = combine_heads(head_outs)
    return matmul(concat, Wo), per_head_weights

In [27]:
def grouped_query_attention(X, Wq, Wk, Wv, Wo, n_heads, n_kv_heads):
    Q = matmul(X, Wq)
    K = matmul(X, Wk)
    V = matmul(X, Wv)
    Qh = split_heads(Q, n_heads)
    Kh_small = split_heads(K, n_kv_heads)
    Vh_small = split_heads(V, n_kv_heads)
    repeat = n_heads // n_kv_heads
    Kh = [Kh_small[i // repeat] for i in range(n_heads)]
    Vh = [Vh_small[i // repeat] for i in range(n_heads)]

    head_outs = []
    for q, k, v in zip(Qh, Kh, Vh):
        o, w = scaled_dot_product_attention(q, k, v)
        head_outs.append(o)
    concat = combine_heads(head_outs)
    return matmul(concat, Wo)